In [11]:
"""
UCSB Housing Scraper - Customizable Version
Modify the FIELDS_TO_EXTRACT dictionary to specify exactly what you want to scrape
"""
import json
import csv
import time
import re
from datetime import datetime
from typing import List, Dict, Optional
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.options import Options


# ============================================================================
# CUSTOMIZE THIS: Choose which fields you want to extract
# Set to True for fields you want, False for fields you don't want
# ============================================================================
FIELDS_TO_EXTRACT = {
    # Basic Info
    'property_id': True,
    'listing_url': True,
    'title': True,
    'address': True,
    
    # Details Section (from the "Details" area on the page)
    'listing_id': True,
    'campus_name': True,
    'price_from': True,
    'bedrooms': True,
    'bathrooms': True,
    'sq_footage': True,
    'distance_from_campus': False,  # Set to False if you don't want this
    
    # Unit Information (from Units table)
    'units': True,  # Includes all unit details (beds, baths, price, availability)
    'num_units': True,
    
    # Parking Information
    'parking': True,  # Includes all parking details
    
    # Property Features
    'description': True,
    'exceptional_features': True,  # Like "On Bus Line"
    'utilities_included': True,  # What utilities are included in rent
    'laundry': True,  # Laundry information
    'additional_features': True,  # All other features (microwave, patio, etc.)
    
    # Metadata
    'scraped_at': True,
}


class CustomizableUCSBScraper:
    def __init__(self, fields_config: Dict = None):
        """Initialize scraper with customizable fields"""
        self.base_url = "https://rentallistings.housing.ucsb.edu"
        self.search_url = f"{self.base_url}/off-campus-housing/uscb-offcampus/search"
        self.property_ids = []
        self.listings = []
        self.fields_config = fields_config or FIELDS_TO_EXTRACT
        
        chrome_options = Options()
        chrome_options.add_argument('--window-size=1920,1080')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-dev-shm-usage')
        
        self.driver = webdriver.Chrome(options=chrome_options)
        self.wait = WebDriverWait(self.driver, 10)
        print("Chrome browser opened")
        
        # Print what will be scraped
        print("\nFields to extract:")
        for field, enabled in self.fields_config.items():
            if enabled:
                print(f"  ✓ {field}")
    
    def open_and_wait_for_login(self):
        """Open website and wait for manual login"""
        self.driver.get(self.search_url)
        
        print("\n" + "="*70)
        print("BROWSER OPENED - PLEASE LOGIN IF NEEDED")
        print("="*70)
        print("\nInstructions:")
        print("1. Login if required")
        print("2. Wait for the listings page to load")
        print("3. Press ENTER when you're ready to start scraping")
        print("="*70)
        input("\nPress ENTER to continue...")
        print("\nStarting scraping process...\n")
    
    def collect_property_ids(self) -> List[str]:
        """Collect all property IDs from the search page"""
        print("Step 1: Collecting property IDs from search page...")
        
        # Load all listings first
        self.load_all_listings()
        
        property_ids = []
        
        try:
            links = self.driver.find_elements(By.CSS_SELECTOR, 'a[href*="/listings/"]')
            
            for link in links:
                href = link.get_attribute('href')
                if href and '/listings/' in href:
                    match = re.search(r'/listings/(\d+)', href)
                    if match:
                        prop_id = match.group(1)
                        if prop_id not in property_ids:
                            property_ids.append(prop_id)
            
            print(f"  Found {len(property_ids)} unique property IDs")
            self.property_ids = property_ids
            return property_ids
            
        except Exception as e:
            print(f"Error collecting property IDs: {e}")
            return []
    
    def load_all_listings(self):
        """Click 'Load More' until all listings are visible"""
        print("  Loading all listings...")
        clicks = 0
        max_clicks = 50
        
        while clicks < max_clicks:
            try:
                load_more = self.wait.until(
                    EC.element_to_be_clickable((By.LINK_TEXT, "Load More"))
                )
                self.driver.execute_script("arguments[0].scrollIntoView();", load_more)
                time.sleep(0.5)
                self.driver.execute_script("arguments[0].click();", load_more)
                clicks += 1
                print(f"    Clicked 'Load More' ({clicks})")
                time.sleep(2)
            except TimeoutException:
                print("    All listings loaded")
                break
    
    def scrape_listing_detail(self, property_id: str) -> Optional[Dict]:
        """Visit individual listing page and scrape configured fields"""
        try:
            listing_url = f"{self.search_url}?property={property_id}"
            self.driver.get(listing_url)
            time.sleep(2)
            
            data = {}
            
            # Get the full page text for parsing
            body_text = self.driver.find_element(By.TAG_NAME, 'body').text
            
            # Basic Info
            if self.fields_config.get('property_id'):
                data['property_id'] = property_id
            
            if self.fields_config.get('listing_url'):
                data['listing_url'] = listing_url
            
            if self.fields_config.get('scraped_at'):
                data['scraped_at'] = datetime.now().isoformat()
            
            # Title (usually the address or property name)
            if self.fields_config.get('title'):
                try:
                    title = self.driver.find_element(By.CSS_SELECTOR, 'h2, h1').text
                    data['title'] = title.strip()
                except:
                    data['title'] = 'N/A'
            
            # Address
            if self.fields_config.get('address'):
                data['address'] = self.extract_field(body_text, r'Address[:\s]+(.+?)(?:\n|$)')
            
            # Details Section Fields
            if self.fields_config.get('listing_id'):
                data['listing_id'] = self.extract_field(body_text, r'Listing Id[:\s]+(\d+)')
            
            if self.fields_config.get('campus_name'):
                data['campus_name'] = self.extract_field(body_text, r'Campus Name[:\s]+(.+?)(?:\n|$)')
            
            if self.fields_config.get('price_from'):
                data['price_from'] = self.extract_field(body_text, r'Price From[:\s]+\$?([\d,]+)')
            
            if self.fields_config.get('bedrooms'):
                data['bedrooms'] = self.extract_field(body_text, r'Bedrooms[:\s]+(.+?)(?:\n|$)')
            
            if self.fields_config.get('bathrooms'):
                data['bathrooms'] = self.extract_field(body_text, r'Bathrooms[:\s]+(.+?)(?:\n|$)')
            
            if self.fields_config.get('sq_footage'):
                data['sq_footage'] = self.extract_field(body_text, r'Sq footage[:\s]+([\d,]+)')
            
            if self.fields_config.get('distance_from_campus'):
                data['distance_from_campus'] = self.extract_field(body_text, r'Distance from campus[:\s]+(.+?)(?:\n|$)')
            
            # Units table
            if self.fields_config.get('units') or self.fields_config.get('num_units'):
                units = self.scrape_units_table()
                if units:
                    if self.fields_config.get('units'):
                        data['units'] = units
                    if self.fields_config.get('num_units'):
                        data['num_units'] = len(units)
            
            # Parking information
            if self.fields_config.get('parking'):
                parking = self.scrape_parking_table()
                if parking:
                    data['parking'] = parking
            
            # Description
            if self.fields_config.get('description'):
                description = self.extract_description(body_text)
                if description:
                    data['description'] = description
            
            # Exceptional Features (like "On Bus Line")
            if self.fields_config.get('exceptional_features'):
                features = self.extract_section_items(body_text, "EXCEPTIONAL FEATURES")
                if features:
                    data['exceptional_features'] = features
            
            # Utilities Included
            if self.fields_config.get('utilities_included'):
                utilities = self.extract_section_items(body_text, "UTILITIES INCLUDED IN RENT")
                if utilities:
                    data['utilities_included'] = utilities
            
            # Laundry
            if self.fields_config.get('laundry'):
                laundry = self.extract_section_items(body_text, "LAUNDRY")
                if laundry:
                    data['laundry'] = laundry
            
            # Additional Features
            if self.fields_config.get('additional_features'):
                features = self.extract_section_items(body_text, "ADDITIONAL FEATURES")
                if features:
                    data['additional_features'] = features
            
            return data
            
        except Exception as e:
            print(f"  Error scraping property {property_id}: {e}")
            return None
    
    def extract_field(self, text: str, pattern: str) -> str:
        """Extract a single field using regex pattern"""
        match = re.search(pattern, text, re.IGNORECASE)
        return match.group(1).strip() if match else 'N/A'
    
    def extract_description(self, text: str) -> str:
        """Extract the description section"""
        # Look for DESCRIPTION header and extract text until next header
        match = re.search(r'DESCRIPTION\s+(.+?)(?:\n\s*[A-Z\s]{10,}\n|$)', text, re.DOTALL)
        if match:
            return match.group(1).strip()
        return 'N/A'
    
    def extract_section_items(self, text: str, section_header: str) -> List[str]:
        """Extract items from a section (like EXCEPTIONAL FEATURES)"""
        items = []
        
        # Find the section
        pattern = rf'{section_header}\s+(.+?)(?:\n\s*[A-Z\s]{{10,}}\n|$)'
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        
        if match:
            section_text = match.group(1).strip()
            # Split by newlines and clean up
            for line in section_text.split('\n'):
                line = line.strip()
                if line and not line.isupper():  # Skip section headers
                    items.append(line)
        
        return items
    
    def scrape_units_table(self) -> List[Dict]:
        """Scrape the Units table"""
        units = []
        
        try:
            tables = self.driver.find_elements(By.TAG_NAME, 'table')
            
            for table in tables:
                headers = []
                header_elements = table.find_elements(By.TAG_NAME, 'th')
                
                if not header_elements:
                    continue
                
                # Get headers
                for th in header_elements:
                    header_text = th.text.strip().lower().replace(' ', '_')
                    headers.append(header_text)
                
                # Check if this looks like the units table
                if 'beds' in headers or 'bedrooms' in headers or 'name' in headers:
                    # Get rows
                    rows = table.find_elements(By.TAG_NAME, 'tr')
                    
                    for row in rows[1:]:  # Skip header row
                        cells = row.find_elements(By.TAG_NAME, 'td')
                        
                        if len(cells) >= len(headers):
                            unit = {}
                            for i, header in enumerate(headers):
                                if i < len(cells):
                                    unit[header] = cells[i].text.strip()
                            
                            if unit:
                                units.append(unit)
            
        except Exception as e:
            print(f"    Error scraping units: {e}")
        
        return units
    
    def scrape_parking_table(self) -> List[Dict]:
        """Scrape the Parking Features table"""
        parking = []
        
        try:
            tables = self.driver.find_elements(By.TAG_NAME, 'table')
            
            for table in tables:
                headers = []
                header_elements = table.find_elements(By.TAG_NAME, 'th')
                
                if not header_elements:
                    continue
                
                # Get headers
                for th in header_elements:
                    header_text = th.text.strip().lower().replace(' ', '_')
                    headers.append(header_text)
                
                # Check if this looks like the parking table
                if 'parking_type' in headers or 'parking' in ' '.join(headers):
                    rows = table.find_elements(By.TAG_NAME, 'tr')
                    
                    for row in rows[1:]:  # Skip header row
                        cells = row.find_elements(By.TAG_NAME, 'td')
                        
                        if len(cells) >= len(headers):
                            parking_info = {}
                            for i, header in enumerate(headers):
                                if i < len(cells):
                                    parking_info[header] = cells[i].text.strip()
                            
                            if parking_info:
                                parking.append(parking_info)
            
        except Exception as e:
            print(f"    Error scraping parking: {e}")
        
        return parking
    
    def scrape_all_listings(self):
        """Main method to scrape all listings"""
        property_ids = self.collect_property_ids()
        
        if not property_ids:
            print("No property IDs found!")
            return
        
        print(f"\nStep 2: Scraping detailed info for {len(property_ids)} properties...")
        print("This may take a while...\n")
        
        for i, prop_id in enumerate(property_ids, 1):
            print(f"[{i}/{len(property_ids)}] Scraping property {prop_id}...")
            
            listing_data = self.scrape_listing_detail(prop_id)
            
            if listing_data:
                self.listings.append(listing_data)
                title = listing_data.get('title', listing_data.get('address', 'Unknown'))
                print(f"  ✓ Scraped: {title}")
            else:
                print(f"  ✗ Failed to scrape property {prop_id}")
            
            time.sleep(1)
        
        print(f"\n✓ Scraping complete! Total listings: {len(self.listings)}")
    
    def save_to_json(self, filename: Optional[str] = None):
        """Save to JSON"""
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f'ucsb_custom_{timestamp}.json'
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(self.listings, f, indent=2, ensure_ascii=False)
        print(f"Saved to {filename}")
        return filename
    
    def save_to_csv(self, filename: Optional[str] = None):
        """Save to CSV"""
        if not self.listings:
            print("No data to save")
            return None
        
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f'ucsb_custom_{timestamp}.csv'
        
        # Flatten the data for CSV
        flat_listings = []
        for listing in self.listings:
            flat = {}
            for key, value in listing.items():
                if isinstance(value, (list, dict)):
                    flat[key] = json.dumps(value)
                else:
                    flat[key] = value
            flat_listings.append(flat)
        
        # Get all possible keys
        all_keys = set()
        for listing in flat_listings:
            all_keys.update(listing.keys())
        
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=sorted(all_keys))
            writer.writeheader()
            writer.writerows(flat_listings)
        
        print(f"Saved to {filename}")
        return filename
    
    def close(self):
        """Close browser"""
        self.driver.quit()


def main():
    print("\n" + "="*70)
    print(" UCSB CUSTOMIZABLE HOUSING SCRAPER")
    print("="*70)
    
    scraper = None
    
    try:
        scraper = CustomizableUCSBScraper()
        
        scraper.open_and_wait_for_login()
        scraper.scrape_all_listings()
        
        if scraper.listings:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            json_file = scraper.save_to_json(f'ucsb_custom_{timestamp}.json')
            csv_file = scraper.save_to_csv(f'ucsb_custom_{timestamp}.csv')
            
            print("\n" + "="*70)
            print(" SCRAPING COMPLETE!")
            print("="*70)
            print(f"Total listings: {len(scraper.listings)}")
            print(f"JSON file: {json_file}")
            print(f"CSV file: {csv_file}")
            print("="*70)
            
            print("\nSample listing:")
            print(json.dumps(scraper.listings[0], indent=2))
        else:
            print("\nNo listings were scraped!")
        
        print("\nBrowser will close in 10 seconds...")
        time.sleep(10)
        
    except KeyboardInterrupt:
        print("\n\nInterrupted by user")
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if scraper:
            scraper.close()


if __name__ == "__main__":
    main()


 UCSB CUSTOMIZABLE HOUSING SCRAPER
Chrome browser opened

Fields to extract:
  ✓ property_id
  ✓ listing_url
  ✓ title
  ✓ address
  ✓ listing_id
  ✓ campus_name
  ✓ price_from
  ✓ bedrooms
  ✓ bathrooms
  ✓ sq_footage
  ✓ units
  ✓ num_units
  ✓ parking
  ✓ description
  ✓ exceptional_features
  ✓ utilities_included
  ✓ laundry
  ✓ additional_features
  ✓ scraped_at

BROWSER OPENED - PLEASE LOGIN IF NEEDED

Instructions:
1. Login if required
2. Wait for the listings page to load
3. Press ENTER when you're ready to start scraping



Press ENTER to continue... 



Starting scraping process...

Step 1: Collecting property IDs from search page...
  Loading all listings...
    All listings loaded
  Found 49 unique property IDs

Step 2: Scraping detailed info for 49 properties...
This may take a while...

[1/49] Scraping property 5575...
  ✓ Scraped: UC Santa Barbara Off-Campus Housing
[2/49] Scraping property 719...
  ✓ Scraped: UC Santa Barbara Off-Campus Housing
[3/49] Scraping property 5846...
  ✓ Scraped: UC Santa Barbara Off-Campus Housing
[4/49] Scraping property 5843...
  ✓ Scraped: UC Santa Barbara Off-Campus Housing
[5/49] Scraping property 5842...
  ✓ Scraped: UC Santa Barbara Off-Campus Housing
[6/49] Scraping property 5840...
  ✓ Scraped: UC Santa Barbara Off-Campus Housing
[7/49] Scraping property 5830...
  ✓ Scraped: UC Santa Barbara Off-Campus Housing
[8/49] Scraping property 5796...
  ✓ Scraped: UC Santa Barbara Off-Campus Housing
[9/49] Scraping property 5790...
  ✓ Scraped: Test | Rent College Pads
[10/49] Scraping property 5750.

In [13]:
import pandas as pd

In [23]:
new_df = pd.read_csv('ucsb_custom_20260412_121502.csv')

In [24]:
new_df.head()

,additional_features,address,bathrooms,bedrooms,campus_name,description,exceptional_features,laundry,listing_id,listing_url,price_from,property_id,scraped_at,sq_footage,title,utilities_included
0,"[""View Similar Properties By: Beds | Price"", ""...","6587 Cervantes Rd Goleta, CA 93117",1 Bath,1 Bed,UC Santa Barbara,Welcome to the best of modern living in Isla V...,NaN,"[""access, and beautiful flooring throughout. T...",NaN,https://rentallistings.housing.ucsb.edu/off-ca...,3450,5575,2026-04-12T12:11:50.461875,550.0,UC Santa Barbara Off-Campus Housing,"[""Gas""]"
1,"[""Microwave""]","6640 Pasado Rd, Goleta, CA 93117, United States",2 Bath,3 Bed,UC Santa Barbara,DUPLEX: BOTH A & B AVAILABLE FOR RENT. ALSO ...,"[""Dishwasher""]","[""6/15/26.""]",NaN,https://rentallistings.housing.ucsb.edu/off-ca...,5600,719,2026-04-12T12:11:54.392801,2400.0,UC Santa Barbara Off-Campus Housing,"[""Cable TV""]"
2,"[""View Similar Properties By: Beds | Price"", ""...","6873 Fortuna Rd Goleta, CA 93117",1 Bath,1 Bed,UC Santa Barbara,Furnished room( for single occupancy only) sha...,NaN,NaN,NaN,https://rentallistings.housing.ucsb.edu/off-ca...,1100,5846,2026-04-12T12:11:58.666037,NaN,UC Santa Barbara Off-Campus Housing,"[""Gas""]"
3,"[""View Similar Properties By: Beds | Price"", ""...","6773 Estero Rd Goleta, CA 93117",2 - 3 Bath,5 Bed,UC Santa Barbara,BEAUTIFULLY RENOVATED HOUSE TO ENJOY FOR YOUR ...,NaN,NaN,NaN,https://rentallistings.housing.ucsb.edu/off-ca...,12500,5843,2026-04-12T12:12:02.206787,1121.0,UC Santa Barbara Off-Campus Housing,"[""Gas""]"
4,"[""View Similar Properties By: Beds | Price"", ""...","6555 Segovia Rd Goleta, CA 93117",3.5 Bath,7 Bed,UC Santa Barbara,"THIS IS AN AMAZING 2 STORY, 7 BEDROOM 3.5 BATH...",NaN,NaN,NaN,https://rentallistings.housing.ucsb.edu/off-ca...,16800,5842,2026-04-12T12:12:06.221406,1600.0,UC Santa Barbara Off-Campus Housing,"[""Gas""]"


In [25]:
old_df = pd.read_csv('ucsb_cleaned_listings.csv')

In [26]:
old_df.head()

,address,bathrooms,bedrooms,price_from,distance_to_ucsb_miles
0,"759 Embarcadero del Mar, Goleta, CA 93117, Uni...",1 Bath,1 - 2 Bed,3450.0,0.51
1,"4781 Avalon Ave Santa Barbara, CA 93110",1 Bath,1 Bed,2600.0,3.60
2,"6587 Cervantes Rd Goleta, CA 93117",1 Bath,1 Bed,3450.0,0.53
3,"6777 Del Playa Dr Isla Vista, CA 93117",2 Bath,3 - 5 Bed,10000.0,1.00
4,"6561 Del Playa Dr Goleta, CA 93117",1 - 2 Bath,S - 3 Bed,7000.0,0.56


In [31]:
old_df.shape

(119, 5)

In [27]:
filtered_df = new_df[~new_df["address"].isin(old_df["address"])]

In [29]:
filtered_df.shape

(19, 16)

In [30]:
filtered_df.head()

,additional_features,address,bathrooms,bedrooms,campus_name,description,exceptional_features,laundry,listing_id,listing_url,price_from,property_id,scraped_at,sq_footage,title,utilities_included
1,"[""Microwave""]","6640 Pasado Rd, Goleta, CA 93117, United States",2 Bath,3 Bed,UC Santa Barbara,DUPLEX: BOTH A & B AVAILABLE FOR RENT. ALSO ...,"[""Dishwasher""]","[""6/15/26.""]",NaN,https://rentallistings.housing.ucsb.edu/off-ca...,5600,719,2026-04-12T12:11:54.392801,2400.0,UC Santa Barbara Off-Campus Housing,"[""Cable TV""]"
2,"[""View Similar Properties By: Beds | Price"", ""...","6873 Fortuna Rd Goleta, CA 93117",1 Bath,1 Bed,UC Santa Barbara,Furnished room( for single occupancy only) sha...,NaN,NaN,NaN,https://rentallistings.housing.ucsb.edu/off-ca...,1100,5846,2026-04-12T12:11:58.666037,NaN,UC Santa Barbara Off-Campus Housing,"[""Gas""]"
3,"[""View Similar Properties By: Beds | Price"", ""...","6773 Estero Rd Goleta, CA 93117",2 - 3 Bath,5 Bed,UC Santa Barbara,BEAUTIFULLY RENOVATED HOUSE TO ENJOY FOR YOUR ...,NaN,NaN,NaN,https://rentallistings.housing.ucsb.edu/off-ca...,12500,5843,2026-04-12T12:12:02.206787,1121.0,UC Santa Barbara Off-Campus Housing,"[""Gas""]"
4,"[""View Similar Properties By: Beds | Price"", ""...","6555 Segovia Rd Goleta, CA 93117",3.5 Bath,7 Bed,UC Santa Barbara,"THIS IS AN AMAZING 2 STORY, 7 BEDROOM 3.5 BATH...",NaN,NaN,NaN,https://rentallistings.housing.ucsb.edu/off-ca...,16800,5842,2026-04-12T12:12:06.221406,1600.0,UC Santa Barbara Off-Campus Housing,"[""Gas""]"
6,"[""View Similar Properties By: Beds | Price"", ""...","280 Royal Linda Dr, Goleta, CA 93117, United S...",1 Bath,1 Bed,UC Santa Barbara,"There are two bedrooms downstairs, each room h...",NaN,NaN,NaN,https://rentallistings.housing.ucsb.edu/off-ca...,1500,5830,2026-04-12T12:12:14.038828,245.0,UC Santa Barbara Off-Campus Housing,"[""Gas""]"


In [41]:
filtered_df = filtered_df.drop(columns=[
    'additional_features', 'description', 'exceptional_features',
    'laundry', 'listing_id', 'listing_url', 'property_id',
    'scraped_at', 'sq_footage', 'title', 'utilities_included', 'campus_name'
], errors='ignore')

In [42]:
filtered_df.columns

Index(['address', 'bathrooms', 'bedrooms', 'price_from'], dtype='object')

In [43]:
filtered_df.head()

,address,bathrooms,bedrooms,price_from
1,"6640 Pasado Rd, Goleta, CA 93117, United States",2 Bath,3 Bed,5600
2,"6873 Fortuna Rd Goleta, CA 93117",1 Bath,1 Bed,1100
3,"6773 Estero Rd Goleta, CA 93117",2 - 3 Bath,5 Bed,12500
4,"6555 Segovia Rd Goleta, CA 93117",3.5 Bath,7 Bed,16800
6,"280 Royal Linda Dr, Goleta, CA 93117, United S...",1 Bath,1 Bed,1500


In [52]:
pip install openrouteservice

Note: you may need to restart the kernel to use updated packages.


In [66]:
# Now calcuating the geo code for the distance to ucsb column
import openrouteservice
import time

client = openrouteservice.Client(key="")


In [71]:
ucsb_coords = [-119.8489, 34.4139]  # [lon, lat]

In [72]:
def get_drive_distance(address):
    try:
        # geocode address → coordinates
        geocode = client.pelias_search(text=address)
        coords = geocode['features'][0]['geometry']['coordinates']  # [lon, lat]

        # get driving route
        route = client.directions(
            coordinates=[ucsb_coords, coords],
            profile='driving-car'
        )

        distance_meters = route['routes'][0]['summary']['distance']
        distance_miles = distance_meters / 1609.34

        time.sleep(1)  # avoid rate limits
        return distance_miles

    except:
        return None

In [73]:
filtered_df["distance_to_ucsb_miles"] = filtered_df["address"].apply(get_drive_distance)

In [92]:
filtered_df.head()

,address,bathrooms,bedrooms,price_from,distance_to_ucsb_miles
1,"6640 Pasado Rd, Goleta, CA 93117, United States",2 Bath,3 Bed,5600,1.2
2,"6873 Fortuna Rd Goleta, CA 93117",1 Bath,1 Bed,1100,1.6
3,"6773 Estero Rd Goleta, CA 93117",2 - 3 Bath,5 Bed,12500,1.6
4,"6555 Segovia Rd Goleta, CA 93117",3.5 Bath,7 Bed,16800,0.8
6,"280 Royal Linda Dr, Goleta, CA 93117, United S...",1 Bath,1 Bed,1500,3.8


In [93]:
filtered_df["distance_to_ucsb_miles"] = filtered_df["distance_to_ucsb_miles"].round(1)
filtered_df

,address,bathrooms,bedrooms,price_from,distance_to_ucsb_miles
1,"6640 Pasado Rd, Goleta, CA 93117, United States",2 Bath,3 Bed,5600,1.2
2,"6873 Fortuna Rd Goleta, CA 93117",1 Bath,1 Bed,1100,1.6
3,"6773 Estero Rd Goleta, CA 93117",2 - 3 Bath,5 Bed,12500,1.6
4,"6555 Segovia Rd Goleta, CA 93117",3.5 Bath,7 Bed,16800,0.8
6,"280 Royal Linda Dr, Goleta, CA 93117, United S...",1 Bath,1 Bed,1500,3.8
8,"1101 N Market St Milwaukee, WI 53202",1 - 2.5 Bath,1 - 2 Bed,150,2157.1
14,"759 Embarcadero del Mar Isla Vista, CA 93117",1 Bath,1 Bed,1150,0.7
21,"6531 Sabado Tarde Rd, Isla Vista, CA 93117, Un...",2 Bath,2 Bed,5400,1.1
36,"6896 Willowgrove Dr Goleta, CA 93117",1 Bath,1 Bed,1600,1.5
39,"6719 Sabado Tarde Rd Isla Vista, CA 93117",1 Bath,1 Bed,0,1.4


In [95]:
filtered_df = filtered_df.drop(filtered_df.index[5])

In [96]:
filtered_df 

,address,bathrooms,bedrooms,price_from,distance_to_ucsb_miles
1,"6640 Pasado Rd, Goleta, CA 93117, United States",2 Bath,3 Bed,5600,1.2
2,"6873 Fortuna Rd Goleta, CA 93117",1 Bath,1 Bed,1100,1.6
3,"6773 Estero Rd Goleta, CA 93117",2 - 3 Bath,5 Bed,12500,1.6
4,"6555 Segovia Rd Goleta, CA 93117",3.5 Bath,7 Bed,16800,0.8
6,"280 Royal Linda Dr, Goleta, CA 93117, United S...",1 Bath,1 Bed,1500,3.8
14,"759 Embarcadero del Mar Isla Vista, CA 93117",1 Bath,1 Bed,1150,0.7
21,"6531 Sabado Tarde Rd, Isla Vista, CA 93117, Un...",2 Bath,2 Bed,5400,1.1
36,"6896 Willowgrove Dr Goleta, CA 93117",1 Bath,1 Bed,1600,1.5
39,"6719 Sabado Tarde Rd Isla Vista, CA 93117",1 Bath,1 Bed,0,1.4
41,"777 Camino Pescadero Isla Vista, CA 93117",1 Bath,1 Bed,876,0.8


In [97]:
filtered_df.to_csv("/Users/anugrhatamang/Desktop/housing_listings.csv", index=False)